# DL TCN 벤치마크
Purpose: define a safe dual-T4 causal TCN benchmark using the immutable shared validation contract.

> Warning: this is an oracle/sanity-only synthetic-data benchmark. Real accuracy is NOT VERIFIED and this is not medical or diagnostic evidence.

In [ ]:
from __future__ import annotations

import hashlib
import json
import os
import random
from bisect import bisect_right
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Mapping, Sequence

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import torch
import torch.distributed as dist
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import (
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    f1_score,
    recall_score,
    roc_auc_score,
)
from torch.nn.parallel import DistributedDataParallel
from torch.utils.data import DataLoader, Dataset, DistributedSampler

SERIES_ID = "mvp3-oracle-v1"
EXPECTED_SPLIT_COUNTS = {"train": 24, "validation": 6, "locked_test": 6}
DATA_STATUS = "oracle/sanity"
REAL_ACCURACY_STATUS = "NOT VERIFIED"
DEVICE_SYNCHRONIZATION_STATUS = "NOT_AVAILABLE_TRUTH_ONLY"
RUN_TRAINING = False
RUN_LOCKED_TEST = False

SEQUENCE_OUTPUT_ROOT = Path("/kaggle/working/goal15_dl_sequences")
ML_VIEW_ROOT = Path('/kaggle/working/goal15_ml_view')
DL_BENCHMARK_OUTPUT_ROOT = Path('/kaggle/working/goal15_dl_tcn_benchmark')
ML_BENCHMARK_OUTPUT_ROOT = Path('/kaggle/working/goal15_ml_benchmark')
SEQUENCE_LENGTHS_SECONDS = (300, 600)
PATTERN_TARGET = "pattern_binary"
ONSET_EVENT_TARGET = "event_binary"
STAGE_TARGET = 'stage_code'
STAGE_CODES = ('LOW', 'MEDIUM', 'HIGH', 'DECREASING', 'RECOVERY')
BEHAVIOR_CODES = (
    'ear_covering',
    'exit_attempt',
    'head_turn_away',
    'motion_freeze',
    'movement_reduction',
    'repetitive_body_movement',
    'repetitive_hand_movement',
    'repetitive_object_contact',
    'sustained_pressure_or_contact',
    'withdrawal_movement',
)
CAUSAL_FACTORS = (
    'autonomic_arousal', 'motor_activation', 'cognitive_load',
    'sleep_pressure', 'sensory_context', 'recovery_capacity', 'social_context',
)
ROLLING_STATISTICS = ('mean', 'std', 'slope')
ROLLING_WINDOWS_SECONDS = (5, 15, 30, 60, 180, 300)
TIME_FEATURE_COLUMNS = ('time_sin', 'time_cos', 'weekday_sin', 'weekday_cos', 'is_awake')
CONTEXT_FEATURE_COLUMNS = (
    'context__sleep', 'context__transition', 'context__meal_context',
    'context__focused_task', 'context__moderate_activity',
    'context__light_activity', 'context__wake_rest', 'context__sedentary_activity',
)
APPROVED_CONTEXTS = (
    'sleep', 'transition', 'meal_context', 'focused_task',
    'moderate_activity', 'light_activity', 'wake_rest', 'sedentary_activity',
)
ALLOWED_FEATURE_COLUMNS = tuple(
    [
        feature
        for factor in CAUSAL_FACTORS
        for feature in (
            f'{factor}__robust_z',
            *(
                f'{factor}__{statistic}_{window_seconds}s'
                for window_seconds in ROLLING_WINDOWS_SECONDS
                for statistic in ROLLING_STATISTICS
            ),
        )
    ]
    + list(TIME_FEATURE_COLUMNS)
    + list(CONTEXT_FEATURE_COLUMNS)
)
PREDICTION_COLUMNS = [
    'model_family', 'model_name', 'series_id', 'dataset_id', 'run_id',
    'person_key', 'canonical_time', 'split_role', 'target', 'label',
    'probability', 'threshold',
]
METRIC_COLUMNS = [
    'model_family', 'model_name', 'series_id', 'split_role', 'target',
    'metric', 'value', 'support', 'data_status',
]
SEQUENCE_INDEX_COLUMNS = {
    'person_key', 'run_id', 'dataset_id', 'context', 'split_role',
    PATTERN_TARGET, ONSET_EVENT_TARGET, 'hard_negative', STAGE_TARGET,
    *BEHAVIOR_CODES, 'window_start', 'window_end', 'prediction_time',
    'length_seconds', 'window_id', 'sample_type',
}
SEED = 20260728
WANDB_PROJECT = 'multisensor-goal15-benchmark'
WANDB_GROUP = 'deep-learning-tcn'
WANDB_TAGS = ['oracle-sanity', 'mvp3', 'split-24-6-6', 'not-real-verified']


## 1. 입력 무결성 검증
Task 4가 만든 manifest, train-only 정규화, 여섯 sequence index와 원본 full causal timeline을 독립적으로 검증합니다.

In [ ]:
@dataclass(frozen=True)
class VerifiedSequenceInputs:
    source_dataset_hash: str
    split_hash: str
    index_paths: Mapping[str, Path]
    timeline_paths: Mapping[str, Path]
    normalization: Mapping[str, Any]


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


def _require_sha256(value: Any, field: str) -> str:
    if not isinstance(value, str) or len(value) != 64:
        raise ValueError(f'invalid SHA-256 for {field}')
    try:
        int(value, 16)
    except ValueError as exc:
        raise ValueError(f'invalid SHA-256 for {field}') from exc
    return value


def _safe_child(root: Path, name: Any, field: str) -> Path:
    if not isinstance(name, str) or not name or Path(name).name != name:
        raise ValueError(f'invalid relative path for {field}')
    path = root / name
    if not path.is_file():
        raise FileNotFoundError(f'missing {field}: {path}')
    return path


def _validate_timeline_schema(parquet: pq.ParquetFile, split_role: str) -> None:
    columns = list(parquet.schema_arrow.names)
    ordered_features = [column for column in columns if column in ALLOWED_FEATURE_COLUMNS]
    if ordered_features != list(ALLOWED_FEATURE_COLUMNS):
        raise ValueError(f'exact feature schema/order mismatch: {split_role}')
    for feature in ALLOWED_FEATURE_COLUMNS:
        data_type = parquet.schema_arrow.field(feature).type
        if not (pa.types.is_boolean(data_type) or pa.types.is_integer(data_type) or pa.types.is_floating(data_type)):
            raise ValueError(f'non-numeric causal feature: {feature}')


def _stream_role_people(parquet: pq.ParquetFile, split_role: str) -> set[str]:
    required = {'person_key', 'split_role'}
    if not required.issubset(parquet.schema_arrow.names):
        raise ValueError(f'timeline identity schema mismatch: {split_role}')
    people: set[str] = set()
    for row_group in range(parquet.num_row_groups):
        frame = parquet.read_row_group(row_group, columns=['person_key', 'split_role']).to_pandas()
        if frame[['person_key', 'split_role']].isna().any().any():
            raise ValueError(f'null timeline identity: {split_role}')
        if not frame['split_role'].eq(split_role).all():
            raise ValueError(f'timeline split binding mismatch: {split_role}')
        if not frame['person_key'].map(lambda value: isinstance(value, str) and bool(value.strip())).all():
            raise ValueError(f'invalid person identity: {split_role}')
        people.update(frame['person_key'])
    return people


def _verify_sequence_index(
    path: Path,
    split_role: str,
    expected_length: int,
    metadata: Mapping[str, Any],
) -> None:
    parquet = pq.ParquetFile(path)
    if metadata.get('row_count') != parquet.metadata.num_rows:
        raise ValueError(f'sequence row_count mismatch: {path.name}')
    if not SEQUENCE_INDEX_COLUMNS.issubset(parquet.schema_arrow.names):
        raise ValueError(f'sequence index schema mismatch: {path.name}')
    allowed_sample_types = (
        {'positive_centered', 'hard_negative', 'matched_baseline'}
        if split_role == 'train'
        else {'sliding'}
    )
    seen_people: set[str] = set()
    previous_order: tuple[str, str, str, int, int, str] | None = None
    for row_group in range(parquet.num_row_groups):
        frame = parquet.read_row_group(
            row_group,
            columns=list(SEQUENCE_INDEX_COLUMNS),
        ).to_pandas()
        identity_columns = ['person_key', 'run_id', 'dataset_id', 'context', 'window_id', 'sample_type']
        if frame[identity_columns].isna().any().any():
            raise ValueError(f'null sequence identity: {path.name}')
        for column in identity_columns:
            if not frame[column].map(lambda value: isinstance(value, str) and bool(value) and value == value.strip()).all():
                raise ValueError(f'invalid sequence identity {column}: {path.name}')
        if not frame['context'].isin(APPROVED_CONTEXTS).all():
            raise ValueError(f'invalid context domain: {path.name}')
        if not frame['split_role'].eq(split_role).all():
            raise ValueError(f'sequence index split mismatch: {path.name}')
        if not frame['sample_type'].isin(allowed_sample_types).all():
            raise ValueError(f'sampled or wrong sequence manifest: {path.name}')
        if not frame['length_seconds'].eq(expected_length).all():
            raise ValueError(f'sequence length mismatch: {path.name}')
        binary_columns = [PATTERN_TARGET, ONSET_EVENT_TARGET, 'hard_negative', *BEHAVIOR_CODES]
        if frame[binary_columns].isna().any().any() or not all(frame[column].isin((0, 1)).all() for column in binary_columns):
            raise ValueError(f'sequence labels must be exact binary: {path.name}')
        if frame[STAGE_TARGET].isna().any() or not frame[STAGE_TARGET].isin(('NO_EVENT', *STAGE_CODES)).all():
            raise ValueError(f'invalid stage labels: {path.name}')
        expected_pattern = frame[STAGE_TARGET].ne('NO_EVENT').astype(np.int8)
        if not frame[PATTERN_TARGET].astype(np.int8).eq(expected_pattern).all():
            raise ValueError(f'pattern/stage mismatch: {path.name}')
        if (frame[PATTERN_TARGET].eq(1) & frame['hard_negative'].eq(1)).any():
            raise ValueError('pattern and hard_negative cannot overlap')
        behavior_positive = frame.loc[:, list(BEHAVIOR_CODES)].eq(1).any(axis=1)
        if (behavior_positive & frame[PATTERN_TARGET].eq(0) & frame['hard_negative'].eq(0)).any():
            raise ValueError('behavior-positive ordinary baseline is forbidden')
        parsed_times: dict[str, pd.Series] = {}
        for column in ('window_start', 'window_end', 'prediction_time'):
            times = pd.to_datetime(frame[column], errors='raise')
            if not isinstance(times.dtype, pd.DatetimeTZDtype) or str(times.dtype.tz) != 'UTC':
                raise ValueError(f'sequence time must be timezone-aware UTC: {column}')
            parsed_times[column] = times
        if not parsed_times['window_end'].eq(parsed_times['prediction_time']).all():
            raise ValueError(f'window_end must equal prediction_time: {path.name}')
        expected_start = parsed_times['prediction_time'] - pd.Timedelta(seconds=expected_length - 1)
        if not parsed_times['window_start'].eq(expected_start).all():
            raise ValueError(f'causal window_start mismatch: {path.name}')
        for offset, row in enumerate(frame.itertuples(index=False)):
            prediction_time = pd.Timestamp(parsed_times['prediction_time'].iloc[offset])
            identity = f'{row.person_key}|{row.run_id}|{row.dataset_id}|{row.context}|{prediction_time.isoformat()}|{expected_length}'
            expected_window_id = hashlib.sha256(identity.encode()).hexdigest()
            if row.window_id != expected_window_id:
                raise ValueError('deterministic window_id mismatch')
            order = (row.person_key, row.run_id, row.dataset_id, expected_length, prediction_time.value, row.context)
            if previous_order is not None and order <= previous_order:
                raise ValueError(f'sequence index is not globally ordered and unique: {path.name}')
            previous_order = order
        seen_people.update(frame['person_key'])
    if split_role != 'train' and len(seen_people) != EXPECTED_SPLIT_COUNTS[split_role]:
        raise ValueError(f'full evaluation membership mismatch: {split_role}')


def verify_sequence_inputs(
    sequence_root: Path = SEQUENCE_OUTPUT_ROOT,
    timeline_root: Path = ML_VIEW_ROOT,
) -> VerifiedSequenceInputs:
    sequence_manifest_path = sequence_root / 'sequence_manifest.json'
    timeline_manifest_path = timeline_root / 'view_manifest.json'
    if not sequence_manifest_path.is_file() or not timeline_manifest_path.is_file():
        raise FileNotFoundError('sequence and full timeline manifests are required')
    sequence_manifest = json.loads(sequence_manifest_path.read_text())
    timeline_manifest = json.loads(timeline_manifest_path.read_text())
    for manifest in (sequence_manifest, timeline_manifest):
        if manifest.get('series_id') != SERIES_ID or manifest.get('data_status') != DATA_STATUS:
            raise ValueError('shared Dataset identity/status mismatch')
    source_dataset_hash = _require_sha256(sequence_manifest.get('source_dataset_hash'), 'source_dataset_hash')
    split_hash = _require_sha256(sequence_manifest.get('split_hash'), 'split_hash')
    if source_dataset_hash != _require_sha256(timeline_manifest.get('source_dataset_hash'), 'timeline source_dataset_hash'):
        raise ValueError('source_dataset_hash mismatch')
    if split_hash != _require_sha256(timeline_manifest.get('split_hash'), 'timeline split_hash'):
        raise ValueError('split_hash mismatch')

    normalization_metadata = sequence_manifest.get('normalization')
    if not isinstance(normalization_metadata, dict):
        raise ValueError('normalization metadata is required')
    normalization_path = _safe_child(sequence_root, normalization_metadata.get('path'), 'normalization')
    if sha256_file(normalization_path) != _require_sha256(normalization_metadata.get('sha256'), 'normalization'):
        raise ValueError('normalization hash mismatch')
    normalization = json.loads(normalization_path.read_text())
    if normalization.get('fit_split_role') != 'train':
        raise ValueError('normalization must be fitted on train only')
    if normalization.get('series_id') != SERIES_ID or normalization.get('source_hash') != source_dataset_hash:
        raise ValueError('normalization source identity mismatch')
    feature_statistics = normalization.get('features')
    if not isinstance(feature_statistics, dict) or set(feature_statistics) != set(ALLOWED_FEATURE_COLUMNS):
        raise ValueError('normalization exact feature schema mismatch')
    for feature in ALLOWED_FEATURE_COLUMNS:
        statistic = feature_statistics[feature]
        if not isinstance(statistic, dict) or set(statistic) != {'median', 'iqr'}:
            raise ValueError(f'invalid train normalization: {feature}')
        values = np.asarray([statistic['median'], statistic['iqr']], dtype=np.float64)
        if not np.isfinite(values).all() or values[1] <= 0:
            raise ValueError(f'invalid train normalization values: {feature}')

    timeline_files = timeline_manifest.get('dl_timeline_files')
    if not isinstance(timeline_files, dict) or set(timeline_files) != set(EXPECTED_SPLIT_COUNTS):
        raise ValueError('full causal timeline roles are incomplete')
    timeline_paths: dict[str, Path] = {}
    role_people: dict[str, set[str]] = {}
    expected_feature_types: tuple[str, ...] | None = None
    for split_role, metadata in timeline_files.items():
        if metadata.get('view_kind') != 'full_causal_timeline' or metadata.get('sampled') is not False:
            raise ValueError(f'sampled full timeline is forbidden: {split_role}')
        if metadata.get('feature_columns') != list(ALLOWED_FEATURE_COLUMNS):
            raise ValueError(f'timeline feature schema/order mismatch: {split_role}')
        path = _safe_child(timeline_root, metadata.get('path'), f'{split_role} timeline')
        if sha256_file(path) != _require_sha256(metadata.get('sha256'), f'{split_role} timeline'):
            raise ValueError(f'timeline hash mismatch: {split_role}')
        parquet = pq.ParquetFile(path)
        if metadata.get('row_count') != parquet.metadata.num_rows:
            raise ValueError(f'timeline row_count mismatch: {split_role}')
        if metadata.get('columns') != list(parquet.schema_arrow.names):
            raise ValueError(f'timeline schema metadata mismatch: {split_role}')
        _validate_timeline_schema(parquet, split_role)
        feature_types = tuple(str(parquet.schema_arrow.field(feature).type) for feature in ALLOWED_FEATURE_COLUMNS)
        if expected_feature_types is None:
            expected_feature_types = feature_types
        elif feature_types != expected_feature_types:
            raise ValueError(f'feature type schema mismatch: {split_role}')
        role_people[split_role] = _stream_role_people(parquet, split_role)
        timeline_paths[split_role] = path
    counts = {role: len(people) for role, people in role_people.items()}
    if counts != EXPECTED_SPLIT_COUNTS:
        raise ValueError(f'person count mismatch: {counts}')
    roles = list(EXPECTED_SPLIT_COUNTS)
    if any(role_people[roles[left]] & role_people[roles[right]] for left in range(3) for right in range(left + 1, 3)):
        raise ValueError('person leakage across split roles')

    sequence_files = sequence_manifest.get('files')
    expected_files = {
        f'{role}_{length}'
        for role in EXPECTED_SPLIT_COUNTS
        for length in SEQUENCE_LENGTHS_SECONDS
    }
    if not isinstance(sequence_files, dict) or set(sequence_files) != expected_files:
        raise ValueError('sequence manifest must declare six exact role/length files')
    index_paths: dict[str, Path] = {}
    for name, metadata in sequence_files.items():
        split_role, length_text = name.rsplit('_', 1)
        if int(length_text) not in SEQUENCE_LENGTHS_SECONDS:
            raise ValueError(f'wrong sequence manifest length: {name}')
        path = _safe_child(sequence_root, metadata.get('path'), name)
        if sha256_file(path) != _require_sha256(metadata.get('sha256'), name):
            raise ValueError(f'sequence index hash mismatch: {name}')
        _verify_sequence_index(path, split_role, int(length_text), metadata)
        index_paths[name] = path
    return VerifiedSequenceInputs(source_dataset_hash, split_hash, index_paths, timeline_paths, normalization)


## 2. 지연 시퀀스 데이터셋
인덱스와 full timeline을 row-group 단위로만 읽고, Task 4의 train 통계로만 정규화합니다.

In [ ]:
class Goal15SequenceDataset(Dataset):
    def __init__(
        self,
        index_path: Path,
        timeline_path: Path,
        normalization: Mapping[str, Any],
        split_role: str,
    ) -> None:
        if split_role not in EXPECTED_SPLIT_COUNTS:
            raise ValueError(f'unknown split role: {split_role}')
        if normalization.get('fit_split_role') != 'train':
            raise ValueError('only train-fitted normalization is accepted')
        if list(normalization.get('features', {}).keys()) != sorted(ALLOWED_FEATURE_COLUMNS):
            if set(normalization.get('features', {})) != set(ALLOWED_FEATURE_COLUMNS):
                raise ValueError('normalization feature schema mismatch')
        self.split_role = split_role
        self.index_file = pq.ParquetFile(index_path)
        self.timeline_file = pq.ParquetFile(timeline_path)
        self.normalization = normalization
        self._index_ends = np.cumsum([
            self.index_file.metadata.row_group(index).num_rows
            for index in range(self.index_file.num_row_groups)
        ]).tolist()
        self._index_cache: tuple[int, pd.DataFrame] | None = None
        self._timeline_catalog = self._catalog_timeline_row_groups()

    def _catalog_timeline_row_groups(self) -> dict[str, list[tuple[int, pd.Timestamp, pd.Timestamp]]]:
        catalog: dict[str, list[tuple[int, pd.Timestamp, pd.Timestamp]]] = {}
        columns = ['dataset_id', 'canonical_time', 'split_role']
        for row_group in range(self.timeline_file.num_row_groups):
            frame = self.timeline_file.read_row_group(row_group, columns=columns).to_pandas()
            if frame.empty or not frame['split_role'].eq(self.split_role).all():
                raise ValueError('timeline row group role mismatch')
            if frame['dataset_id'].nunique(dropna=False) != 1:
                raise ValueError('timeline row group must bind one dataset_id')
            times = pd.to_datetime(frame['canonical_time'], utc=True, errors='raise')
            if times.duplicated().any() or not times.is_monotonic_increasing:
                raise ValueError('timeline row group time must be unique and ordered')
            dataset_id = str(frame['dataset_id'].iloc[0])
            catalog.setdefault(dataset_id, []).append((row_group, times.iloc[0], times.iloc[-1]))
        return catalog

    def __len__(self) -> int:
        return int(self.index_file.metadata.num_rows)

    def _index_row(self, item: int) -> pd.Series:
        if item < 0:
            item += len(self)
        if item < 0 or item >= len(self):
            raise IndexError(item)
        row_group = bisect_right(self._index_ends, item)
        start = 0 if row_group == 0 else self._index_ends[row_group - 1]
        if self._index_cache is None or self._index_cache[0] != row_group:
            self._index_cache = (row_group, self.index_file.read_row_group(row_group).to_pandas())
        return self._index_cache[1].iloc[item - start]

    def _window_frame(self, index_row: pd.Series) -> pd.DataFrame:
        dataset_id = str(index_row['dataset_id'])
        window_start = pd.Timestamp(index_row['window_start'])
        window_end = pd.Timestamp(index_row['window_end'])
        columns = [
            'person_key', 'run_id', 'dataset_id', 'canonical_time',
            *ALLOWED_FEATURE_COLUMNS,
        ]
        pieces: list[pd.DataFrame] = []
        for row_group, first_time, last_time in self._timeline_catalog.get(dataset_id, []):
            if last_time < window_start or first_time > window_end:
                continue
            frame = self.timeline_file.read_row_group(row_group, columns=columns).to_pandas()
            times = pd.to_datetime(frame['canonical_time'], utc=True, errors='raise')
            pieces.append(frame.loc[times.between(window_start, window_end)])
        if not pieces:
            raise ValueError(f'window has no timeline rows: {index_row["window_id"]}')
        window = pd.concat(pieces, ignore_index=True).sort_values('canonical_time')
        expected_length = int(index_row['length_seconds'])
        if len(window) != expected_length:
            raise ValueError(f'window length mismatch: {index_row["window_id"]}')
        for key in ('person_key', 'run_id', 'dataset_id'):
            if not window[key].eq(index_row[key]).all():
                raise ValueError(f'window crosses {key}')
        times = pd.to_datetime(window['canonical_time'], utc=True, errors='raise')
        if times.iloc[0] != window_start or times.iloc[-1] != window_end:
            raise ValueError('window boundary mismatch')
        if not times.diff().iloc[1:].eq(pd.Timedelta(seconds=1)).all():
            raise ValueError('window is not causal contiguous 1 Hz')
        return window

    def __getitem__(self, item: int) -> dict[str, Any]:
        row = self._index_row(item)
        window = self._window_frame(row)
        matrix = window.loc[:, list(ALLOWED_FEATURE_COLUMNS)].to_numpy(dtype=np.float32)
        for column_index, feature in enumerate(ALLOWED_FEATURE_COLUMNS):
            statistic = self.normalization['features'][feature]
            matrix[:, column_index] = (matrix[:, column_index] - float(statistic['median'])) / float(statistic['iqr'])
        if not np.isfinite(matrix).all():
            raise ValueError('normalized sequence contains non-finite values')
        behaviors = np.asarray([row[code] for code in BEHAVIOR_CODES], dtype=np.float32)
        stage_index = STAGE_CODES.index(row[STAGE_TARGET]) if row[STAGE_TARGET] in STAGE_CODES else -1
        return {
            'features': torch.from_numpy(matrix),
            'mask': torch.ones(len(matrix), dtype=torch.bool),
            PATTERN_TARGET: torch.tensor(float(row[PATTERN_TARGET]), dtype=torch.float32),
            ONSET_EVENT_TARGET: torch.tensor(float(row[ONSET_EVENT_TARGET]), dtype=torch.float32),
            'hard_negative': torch.tensor(float(row['hard_negative']), dtype=torch.float32),
            STAGE_TARGET: torch.tensor(stage_index, dtype=torch.long),
            'behaviors': torch.from_numpy(behaviors),
            'dataset_id': str(row['dataset_id']),
            'run_id': str(row['run_id']),
            'person_key': str(row['person_key']),
            'canonical_time': str(pd.Timestamp(row['prediction_time'])),
            'split_role': self.split_role,
        }


## 3. Causal TCN과 조건부 손실
왼쪽 패딩만 쓰는 dilated residual backbone과 event, 5-stage, 10-behavior head를 정의합니다.

In [ ]:
class CausalConvBlock(nn.Module):
    def __init__(self, hidden_size: int, kernel_size: int, dilation: int, dropout: float) -> None:
        super().__init__()
        self.left_padding = (kernel_size - 1) * dilation
        self.conv1 = nn.Conv1d(hidden_size, hidden_size, kernel_size, dilation=dilation, padding=0)
        self.conv2 = nn.Conv1d(hidden_size, hidden_size, kernel_size, dilation=dilation, padding=0)
        self.norm1 = nn.GroupNorm(8, hidden_size)
        self.norm2 = nn.GroupNorm(8, hidden_size)
        self.dropout = nn.Dropout(dropout)

    def _causal_conv(self, values: torch.Tensor, convolution: nn.Conv1d) -> torch.Tensor:
        return convolution(F.pad(values, (self.left_padding, 0)))

    def forward(self, values: torch.Tensor) -> torch.Tensor:
        residual = values
        values = self.dropout(F.gelu(self.norm1(self._causal_conv(values, self.conv1))))
        values = self.dropout(F.gelu(self.norm2(self._causal_conv(values, self.conv2))))
        return values + residual


class Goal15TCN(nn.Module):
    def __init__(self, input_size: int, hidden_size: int = 128, dropout: float = 0.1) -> None:
        super().__init__()
        self.input_projection = nn.Conv1d(input_size, hidden_size, kernel_size=1)
        self.blocks = nn.ModuleList([
            CausalConvBlock(hidden_size, kernel_size=3, dilation=dilation, dropout=dropout)
            for dilation in (1, 2, 4, 8, 16, 32)
        ])
        self.event_head = nn.Linear(hidden_size, 1)
        self.stage_head = nn.Linear(hidden_size, 5)
        self.behavior_head = nn.Linear(hidden_size, 10)

    def forward(self, features: torch.Tensor, mask: torch.Tensor) -> dict[str, torch.Tensor]:
        values = self.input_projection(features.transpose(1, 2))
        for block in self.blocks:
            values = block(values)
        time_mask = mask.unsqueeze(1).to(dtype=values.dtype)
        pooled = (values * time_mask).sum(dim=2) / time_mask.sum(dim=2).clamp_min(1.0)
        return {
            'event_logits': self.event_head(pooled).squeeze(-1),
            'stage_logits': self.stage_head(pooled),
            'behavior_logits': self.behavior_head(pooled),
        }


def count_trainable_parameters(model: nn.Module) -> int:
    return sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)


def assert_parameter_budget(model: nn.Module) -> int:
    count = count_trainable_parameters(model)
    if not 500_000 <= count <= 5_000_000:
        raise ValueError(f'TCN parameter budget violation: {count:,}')
    return count


def masked_multitask_loss(
    outputs: Mapping[str, torch.Tensor],
    batch: Mapping[str, torch.Tensor],
    *,
    stage_weight: float = 1.0,
    behavior_weight: float = 1.0,
) -> tuple[torch.Tensor, dict[str, torch.Tensor]]:
    pattern = batch[PATTERN_TARGET].float()
    event_bce = F.binary_cross_entropy_with_logits(outputs['event_logits'], pattern)
    stage_mask = pattern.eq(1)
    stage_loss = (
        F.cross_entropy(outputs['stage_logits'][stage_mask], batch[STAGE_TARGET][stage_mask])
        if stage_mask.any()
        else outputs['stage_logits'].sum() * 0.0
    )
    behavior_labels = batch['behaviors'].float()
    behavior_positive = behavior_labels.eq(1).any(dim=1)
    hard_negative = batch['hard_negative'].eq(1)
    invalid_positive = behavior_positive & ~(stage_mask | hard_negative)
    if invalid_positive.any():
        raise ValueError('behavior-positive rows must be pattern or hard negative')
    behavior_mask = stage_mask | (hard_negative & behavior_positive)
    behavior_loss = (
        F.binary_cross_entropy_with_logits(
            outputs['behavior_logits'][behavior_mask], behavior_labels[behavior_mask]
        )
        if behavior_mask.any()
        else outputs['behavior_logits'].sum() * 0.0
    )
    total = event_bce + stage_weight * stage_loss + behavior_weight * behavior_loss
    return total, {'event_bce': event_bce, 'stage_loss': stage_loss, 'behavior_loss': behavior_loss}


## 4. T4 x2 DDP 안전 게이트와 학습 루프
GPU 검사는 DDP, W&B, Dataset/DataLoader, trainer보다 먼저 실행됩니다.

In [ ]:
def require_exactly_two_cuda_devices() -> None:
    count = torch.cuda.device_count()
    if count != 2:
        raise RuntimeError(f'T4 x2가 필요합니다. 감지된 CUDA 장치: {count}')


def set_deterministic_seed(seed: int, rank: int = 0) -> None:
    effective_seed = seed + rank
    random.seed(effective_seed)
    np.random.seed(effective_seed)
    torch.manual_seed(effective_seed)
    torch.cuda.manual_seed_all(effective_seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def setup_ddp() -> tuple[int, int, int]:
    required = ('LOCAL_RANK', 'RANK', 'WORLD_SIZE')
    missing = [name for name in required if name not in os.environ]
    if missing:
        raise RuntimeError(f'torchrun 환경이 필요합니다. 누락: {missing}')
    local_rank = int(os.environ['LOCAL_RANK'])
    rank = int(os.environ['RANK'])
    world_size = int(os.environ['WORLD_SIZE'])
    if world_size != 2 or local_rank not in (0, 1):
        raise RuntimeError('T4 x2는 torchrun --nproc_per_node=2로 실행해야 합니다.')
    torch.cuda.set_device(local_rank)
    dist.init_process_group(backend='nccl', init_method='env://', rank=rank, world_size=world_size)
    return rank, local_rank, world_size


def cleanup_ddp() -> None:
    if dist.is_available() and dist.is_initialized():
        dist.barrier()
        dist.destroy_process_group()


def login_wandb_from_kaggle_secret() -> bool:
    try:
        from kaggle_secrets import UserSecretsClient
        import wandb

        key = UserSecretsClient().get_secret("WANDB_API_KEY")
        if not key:
            raise RuntimeError('Kaggle W&B secret is empty')
        return bool(wandb.login(key=key, verify=True))
    except Exception as exc:
        print(f'W&B 비활성화: {type(exc).__name__}')
        return False


def _move_batch(batch: Mapping[str, Any], device: torch.device) -> dict[str, Any]:
    return {
        key: value.to(device, non_blocking=True) if torch.is_tensor(value) else value
        for key, value in batch.items()
    }


def train_one_epoch(
    model: DistributedDataParallel,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    scaler: torch.amp.GradScaler,
    device: torch.device,
    *,
    epoch: int,
    sampler: DistributedSampler,
) -> dict[str, float]:
    sampler.set_epoch(epoch)
    model.train()
    running = {'total': 0.0, 'event_bce': 0.0, 'stage_loss': 0.0, 'behavior_loss': 0.0}
    examples = 0
    for raw_batch in loader:
        batch = _move_batch(raw_batch, device)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast(device_type='cuda', dtype=torch.float16):
            outputs = model(batch['features'], batch['mask'])
            loss, components = masked_multitask_loss(outputs, batch)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        batch_size = int(batch['features'].shape[0])
        examples += batch_size
        running['total'] += float(loss.detach()) * batch_size
        for name, value in components.items():
            running[name] += float(value.detach()) * batch_size
    return {name: value / max(examples, 1) for name, value in running.items()}


## 5. 공통 평가와 validation champion
동일 prediction/metric schema를 사용하며 locked test는 champion 선택 후 별도 gate에서만 평가합니다.

In [ ]:
def expected_calibration_error(labels: np.ndarray, probability: np.ndarray, bins: int = 10) -> float:
    value = 0.0
    edges = np.linspace(0.0, 1.0, bins + 1)
    for lower, upper in zip(edges[:-1], edges[1:], strict=True):
        mask = (probability >= lower) & (probability < upper if upper < 1 else probability <= upper)
        if mask.any():
            value += float(mask.mean()) * abs(float(labels[mask].mean()) - float(probability[mask].mean()))
    return value


def _segments(values: np.ndarray) -> list[tuple[int, int]]:
    changes = np.diff(np.pad(values.astype(np.int8), (1, 1)))
    starts = np.flatnonzero(changes == 1)
    ends = np.flatnonzero(changes == -1) - 1
    return list(zip(starts.tolist(), ends.tolist(), strict=True))


def _event_summary(frame: pd.DataFrame) -> tuple[float, float, float]:
    truth_events = detected = false_alerts = 0
    duration_hours = len(frame) / 3600.0
    for _, person in frame.sort_values(['person_key', 'canonical_time']).groupby('person_key', sort=False):
        truth = person['label'].to_numpy(dtype=bool)
        predicted = person['probability'].ge(person['threshold']).to_numpy(dtype=bool)
        events = _segments(truth)
        truth_events += len(events)
        detected += sum(bool(predicted[start:end + 1].any()) for start, end in events)
        false_alerts += len(_segments(predicted & ~truth))
    event_recall = detected / truth_events if truth_events else 0.0
    event_precision = detected / (detected + false_alerts) if detected + false_alerts else 0.0
    event_f1 = 2 * event_precision * event_recall / (event_precision + event_recall) if event_precision + event_recall else 0.0
    return event_recall, false_alerts / duration_hours if duration_hours else 0.0, event_f1


def _binary_metric_rows(predictions: pd.DataFrame, model_name: str, target: str) -> list[dict[str, Any]]:
    labels = predictions['label'].to_numpy(dtype=np.int8)
    probability = predictions['probability'].to_numpy(dtype=np.float64)
    predicted = probability >= predictions['threshold'].to_numpy(dtype=np.float64)
    two_classes = len(np.unique(labels)) == 2
    aucpr = float(average_precision_score(labels, probability)) if two_classes else np.nan
    auroc = float(roc_auc_score(labels, probability)) if two_classes else np.nan
    person_scores = []
    for _, person in predictions.assign(predicted=predicted).groupby('person_key', sort=False):
        person_scores.append((
            f1_score(person['label'], person['predicted'], zero_division=0),
            recall_score(person['label'], person['predicted'], zero_division=0),
        ))
    person_macro_f1 = float(np.mean([score[0] for score in person_scores])) if person_scores else np.nan
    person_macro_recall = float(np.mean([score[1] for score in person_scores])) if person_scores else np.nan
    event_recall, false_alerts_per_hour, event_f1 = _event_summary(predictions) if target == PATTERN_TARGET else (np.nan, np.nan, np.nan)
    values = {
        'aucpr': aucpr,
        'auroc': auroc,
        'event_recall': event_recall,
        'event_f1': event_f1,
        'false_alerts_per_hour': false_alerts_per_hour,
        'row_f1': float(f1_score(labels, predicted, zero_division=0)),
        'row_recall': float(recall_score(labels, predicted, zero_division=0)),
        'brier_score': float(brier_score_loss(labels, probability)),
        'ece': expected_calibration_error(labels, probability),
        'person_macro_f1': person_macro_f1,
        'person_macro_recall': person_macro_recall,
    }
    return [
        {
            'model_family': 'deep_learning_tcn', 'model_name': model_name,
            'series_id': SERIES_ID, 'split_role': predictions['split_role'].iloc[0],
            'target': target, 'metric': metric, 'value': value,
            'support': len(labels), 'data_status': DATA_STATUS,
        }
        for metric, value in values.items()
    ]


def _aggregate_metric_row(
    model_name: str,
    split_role: str,
    target: str,
    metric: str,
    value: float,
    support: int,
) -> dict[str, Any]:
    return {
        'model_family': 'deep_learning_tcn', 'model_name': model_name,
        'series_id': SERIES_ID, 'split_role': split_role, 'target': target,
        'metric': metric, 'value': value, 'support': support, 'data_status': DATA_STATUS,
    }


def compute_metrics_from_predictions(
    predictions: pd.DataFrame,
    *,
    model_name: str,
) -> pd.DataFrame:
    missing = sorted(set(PREDICTION_COLUMNS).difference(predictions.columns))
    if missing or predictions.empty:
        raise ValueError(f'common predictions are empty or incomplete: {missing}')
    split_roles = set(predictions['split_role'])
    if len(split_roles) != 1:
        raise ValueError('metric input must contain exactly one split role')
    split_role = str(next(iter(split_roles)))
    metric_rows: list[dict[str, Any]] = []
    for target, target_rows in predictions.groupby('target', sort=True):
        metric_rows.extend(_binary_metric_rows(target_rows, model_name, target))

    decision_keys = ['dataset_id', 'run_id', 'person_key', 'canonical_time']
    stage_rows = predictions.loc[predictions['target'].str.startswith('stage::')]
    if not stage_rows.empty:
        stage_probability = stage_rows.pivot(index=decision_keys, columns='target', values='probability')
        stage_labels = stage_rows.loc[stage_rows['label'].eq(1)].set_index(decision_keys)['target']
        if len(stage_labels) != len(stage_probability) or not stage_labels.index.is_unique:
            raise ValueError('conditional stage truth must be exactly one class per pattern row')
        stage_truth = stage_labels.loc[stage_probability.index]
        stage_prediction = stage_probability.idxmax(axis=1)
        stage_macro_f1 = float(f1_score(stage_truth, stage_prediction, average='macro', zero_division=0))
        stage_balanced_accuracy = float(balanced_accuracy_score(stage_truth, stage_prediction))
        metric_rows.extend([
            _aggregate_metric_row(model_name, split_role, 'stage::all', 'stage_macro_f1', stage_macro_f1, len(stage_truth)),
            _aggregate_metric_row(model_name, split_role, 'stage::all', 'stage_balanced_accuracy', stage_balanced_accuracy, len(stage_truth)),
        ])

    behavior_rows = predictions.loc[predictions['target'].str.startswith('behavior::')]
    if not behavior_rows.empty:
        behavior_labels = behavior_rows.pivot(index=decision_keys, columns='target', values='label').astype(np.int8)
        behavior_probability = behavior_rows.pivot(index=decision_keys, columns='target', values='probability').loc[behavior_labels.index, behavior_labels.columns]
        flat_labels = behavior_labels.to_numpy().ravel()
        flat_probability = behavior_probability.to_numpy().ravel()
        behavior_micro_aucpr = float(average_precision_score(flat_labels, flat_probability)) if len(np.unique(flat_labels)) == 2 else np.nan
        per_behavior_aucpr = [
            average_precision_score(behavior_labels[column], behavior_probability[column])
            for column in behavior_labels
            if behavior_labels[column].nunique() == 2
        ]
        behavior_macro_aucpr = float(np.mean(per_behavior_aucpr)) if per_behavior_aucpr else np.nan
        metric_rows.extend([
            _aggregate_metric_row(model_name, split_role, 'behavior::all', 'behavior_micro_aucpr', behavior_micro_aucpr, len(behavior_labels)),
            _aggregate_metric_row(model_name, split_role, 'behavior::all', 'behavior_macro_aucpr', behavior_macro_aucpr, len(behavior_labels)),
        ])
    return pd.DataFrame(metric_rows, columns=METRIC_COLUMNS)


def select_validation_threshold(predictions: pd.DataFrame) -> float:
    pattern = predictions.loc[
        predictions['split_role'].eq('validation') & predictions['target'].eq(PATTERN_TARGET)
    ].copy()
    if pattern.empty or len(pattern) != len(predictions.loc[predictions['target'].eq(PATTERN_TARGET)]):
        raise ValueError('threshold selection accepts validation pattern rows only')
    candidates = np.unique(pattern['probability'].to_numpy(dtype=np.float64))
    scores: list[tuple[float, float, float, float]] = []
    for threshold in candidates:
        pattern['threshold'] = float(threshold)
        predicted = pattern['probability'].ge(threshold)
        event_recall, false_alerts_per_hour, _ = _event_summary(pattern)
        scores.append((
            float(f1_score(pattern['label'], predicted, zero_division=0)),
            event_recall,
            -false_alerts_per_hour,
            float(threshold),
        ))
    return max(scores)[3] if scores else 0.5


def evaluate_common_schema(
    model: nn.Module,
    loader: DataLoader,
    device: torch.device,
    *,
    split_role: str,
    model_name: str,
    thresholds: Mapping[str, float] | None = None,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    if split_role not in EXPECTED_SPLIT_COUNTS:
        raise ValueError(f'unknown split role: {split_role}')
    thresholds = dict(thresholds or {})
    rows: list[dict[str, Any]] = []
    model.eval()
    with torch.no_grad():
        for raw_batch in loader:
            batch = _move_batch(raw_batch, device)
            with torch.amp.autocast(device_type='cuda', dtype=torch.float16):
                outputs = model(batch['features'], batch['mask'])
            pattern_probability = torch.sigmoid(outputs['event_logits']).cpu().numpy()
            stage_probability = torch.softmax(outputs['stage_logits'], dim=1).cpu().numpy()
            behavior_probability = torch.sigmoid(outputs['behavior_logits']).cpu().numpy()
            pattern = batch[PATTERN_TARGET].cpu().numpy().astype(np.int8)
            hard_negative = batch['hard_negative'].cpu().numpy().astype(np.int8)
            behaviors = batch['behaviors'].cpu().numpy().astype(np.int8)
            stage_labels = batch[STAGE_TARGET].cpu().numpy().astype(np.int64)
            behavior_positive = behaviors.eq(1).any(axis=1) if hasattr(behaviors, 'eq') else (behaviors == 1).any(axis=1)
            behavior_mask = (pattern == 1) | ((hard_negative == 1) & behavior_positive)
            for item in range(len(pattern)):
                base = {
                    'model_family': 'deep_learning_tcn', 'model_name': model_name,
                    'series_id': SERIES_ID, 'dataset_id': raw_batch['dataset_id'][item],
                    'run_id': raw_batch['run_id'][item], 'person_key': raw_batch['person_key'][item],
                    'canonical_time': raw_batch['canonical_time'][item], 'split_role': split_role,
                }
                rows.append({**base, 'target': PATTERN_TARGET, 'label': int(pattern[item]), 'probability': float(pattern_probability[item]), 'threshold': float(thresholds.get(PATTERN_TARGET, 0.5))})
                if pattern[item] == 1:
                    for stage_index, stage in enumerate(STAGE_CODES):
                        rows.append({**base, 'target': f'stage::{stage}', 'label': int(stage_labels[item] == stage_index), 'probability': float(stage_probability[item, stage_index]), 'threshold': float(thresholds.get(f'stage::{stage}', 0.5))})
                if behavior_mask[item]:
                    for behavior_index, behavior in enumerate(BEHAVIOR_CODES):
                        rows.append({**base, 'target': f'behavior::{behavior}', 'label': int(behaviors[item, behavior_index]), 'probability': float(behavior_probability[item, behavior_index]), 'threshold': float(thresholds.get(f'behavior::{behavior}', 0.5))})
    predictions = pd.DataFrame(rows, columns=PREDICTION_COLUMNS)
    metrics = compute_metrics_from_predictions(predictions, model_name=model_name)
    return predictions, metrics


def select_validation_champion(metrics: pd.DataFrame) -> str:
    locked_test_rows = metrics.loc[metrics['split_role'].eq('locked_test')]
    validation = metrics.loc[metrics['split_role'].eq('validation') & metrics['target'].eq(PATTERN_TARGET)]
    if validation.empty:
        raise ValueError('validation metrics are required for champion selection')
    _ = locked_test_rows
    pivot = validation.pivot_table(index='model_name', columns='metric', values='value', aggfunc='first')
    required = {'aucpr', 'event_recall', 'false_alerts_per_hour', 'ece'}
    if not required.issubset(pivot.columns):
        raise ValueError('validation tie-break metrics are incomplete')
    ordered = pivot.reset_index().sort_values(
        ['aucpr', 'event_recall', 'false_alerts_per_hour', 'ece', 'model_name'],
        ascending=[False, False, True, True, True],
        kind='mergesort',
    )
    return str(ordered.iloc[0]['model_name'])


def write_validation_model_comparison(
    ml_metrics_path: Path,
    dl_metrics_path: Path,
    output_path: Path = DL_BENCHMARK_OUTPUT_ROOT / 'model_comparison_validation.parquet',
) -> Path:
    ml_metrics = pd.read_parquet(ml_metrics_path)
    dl_metrics = pd.read_parquet(dl_metrics_path)
    for name, frame in (('ML', ml_metrics), ('DL', dl_metrics)):
        if not frame['split_role'].eq('validation').all():
            raise ValueError(f'{name} comparison file must contain validation only; locked_test is forbidden')
    comparison = ml_metrics.merge(
        dl_metrics,
        on=['target', 'metric'],
        how='inner',
        suffixes=('_ml', '_dl'),
        validate='many_to_many',
    )
    output_path.parent.mkdir(parents=True, exist_ok=True)
    comparison.to_parquet(output_path, index=False)
    return output_path


## 6. 실행 오케스트레이션
기본값에서는 아무 학습도 시작하지 않습니다. 실제 실행은 Kaggle T4 x2와 torchrun 2-process 환경을 명시적으로 준비한 뒤에만 가능합니다.

In [ ]:
def _gather_frames(frame: pd.DataFrame, world_size: int) -> pd.DataFrame:
    gathered: list[pd.DataFrame | None] = [None] * world_size
    dist.all_gather_object(gathered, frame)
    return pd.concat([item for item in gathered if item is not None], ignore_index=True)


def run_dl_training(epochs: int = 20, batch_size: int = 64) -> None:
    require_exactly_two_cuda_devices()
    rank, local_rank, world_size = setup_ddp()
    device = torch.device('cuda', local_rank)
    try:
        set_deterministic_seed(SEED, rank)
        verified = verify_sequence_inputs()
        wandb_enabled = rank == 0 and login_wandb_from_kaggle_secret()
        if wandb_enabled:
            import wandb

            wandb.init(project=WANDB_PROJECT, group=WANDB_GROUP, tags=WANDB_TAGS, config={'epochs': epochs, 'batch_size': batch_size, 'data_status': DATA_STATUS})
        train_dataset = Goal15SequenceDataset(
            verified.index_paths['train_600'], verified.timeline_paths['train'],
            verified.normalization, 'train',
        )
        validation_dataset = Goal15SequenceDataset(
            verified.index_paths['validation_600'], verified.timeline_paths['validation'],
            verified.normalization, 'validation',
        )
        train_sampler = DistributedSampler(train_dataset, num_replicas=world_size, rank=rank, shuffle=True, seed=SEED, drop_last=False)
        validation_sampler = DistributedSampler(validation_dataset, num_replicas=world_size, rank=rank, shuffle=False, seed=SEED, drop_last=False)
        train_loader = DataLoader(train_dataset, batch_size=batch_size, sampler=train_sampler, num_workers=2, pin_memory=True)
        validation_loader = DataLoader(validation_dataset, batch_size=batch_size, sampler=validation_sampler, num_workers=2, pin_memory=True)
        model = Goal15TCN(input_size=len(ALLOWED_FEATURE_COLUMNS)).to(device)
        parameter_count = assert_parameter_budget(model)
        model = DistributedDataParallel(model, device_ids=[local_rank], output_device=local_rank)
        optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
        scaler = torch.amp.GradScaler('cuda')
        for epoch in range(epochs):
            train_sampler.set_epoch(epoch)
            losses = train_one_epoch(model, train_loader, optimizer, scaler, device, epoch=epoch, sampler=train_sampler)
            if rank == 0 and wandb_enabled:
                wandb.log({f'train/{name}': value for name, value in losses.items()}, step=epoch)
        local_predictions, _ = evaluate_common_schema(
            model, validation_loader, device, split_role='validation', model_name='causal_tcn',
        )
        validation_predictions = _gather_frames(local_predictions, world_size)
        validation_predictions = validation_predictions.drop_duplicates(
            subset=['dataset_id', 'run_id', 'person_key', 'canonical_time', 'target'],
            keep='first',
        ).reset_index(drop=True)
        selected_threshold: float | None = None
        if rank == 0:
            selected_threshold = select_validation_threshold(validation_predictions)
            validation_predictions.loc[validation_predictions['target'].eq(PATTERN_TARGET), 'threshold'] = selected_threshold
            validation_metrics = compute_metrics_from_predictions(validation_predictions, model_name='causal_tcn')
            champion = select_validation_champion(validation_metrics)
            DL_BENCHMARK_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
            validation_predictions.to_parquet(DL_BENCHMARK_OUTPUT_ROOT / 'validation_predictions.parquet', index=False)
            validation_metrics.to_parquet(DL_BENCHMARK_OUTPUT_ROOT / 'validation_metrics.parquet', index=False)
            (DL_BENCHMARK_OUTPUT_ROOT / 'validation_champion.json').write_text(json.dumps({'model_name': champion, 'parameter_count': parameter_count, 'source_dataset_hash': verified.source_dataset_hash, 'split_hash': verified.split_hash}, indent=2) + '\n')
            if wandb_enabled:
                wandb.log({'validation/champion': champion, 'model/parameter_count': parameter_count})
        threshold_holder = [selected_threshold]
        dist.broadcast_object_list(threshold_holder, src=0)
        selected_threshold = float(threshold_holder[0])
        dist.barrier()
        if RUN_LOCKED_TEST:
            locked_dataset = Goal15SequenceDataset(
                verified.index_paths['locked_test_600'], verified.timeline_paths['locked_test'],
                verified.normalization, 'locked_test',
            )
            locked_sampler = DistributedSampler(locked_dataset, num_replicas=world_size, rank=rank, shuffle=False, seed=SEED, drop_last=False)
            locked_loader = DataLoader(locked_dataset, batch_size=batch_size, sampler=locked_sampler, num_workers=2, pin_memory=True)
            locked_predictions, _ = evaluate_common_schema(
                model, locked_loader, device, split_role='locked_test', model_name='causal_tcn',
                thresholds={PATTERN_TARGET: selected_threshold},
            )
            locked_predictions = _gather_frames(locked_predictions, world_size)
            locked_predictions = locked_predictions.drop_duplicates(
                subset=['dataset_id', 'run_id', 'person_key', 'canonical_time', 'target'],
                keep='first',
            ).reset_index(drop=True)
            if rank == 0:
                locked_metrics = compute_metrics_from_predictions(locked_predictions, model_name='causal_tcn')
                locked_predictions.to_parquet(DL_BENCHMARK_OUTPUT_ROOT / 'locked_test_predictions.parquet', index=False)
                locked_metrics.to_parquet(DL_BENCHMARK_OUTPUT_ROOT / 'locked_test_metrics.parquet', index=False)
        if rank == 0 and wandb_enabled:
            wandb.finish()
    finally:
        cleanup_ddp()


if RUN_TRAINING:
    run_dl_training()
else:
    print('학습 비활성화: RUN_TRAINING=False. GPU, W&B, Dataset/DataLoader를 사용하지 않았습니다.')
